In [1]:
import pandas as pd
import numpy as np
from docplex.mp.model import Model

# Funciones

In [2]:
# --- FUNCIONES AUXILIARES DE FORMATO ---
def formatear_gams(valor, decimales=4, es_na=False):
    if es_na: return "NA"
    if hasattr(valor, 'constant'):
        valor = valor.constant
    valor = float(valor)
    if valor > 1e19: return "+INF"
    if valor < -1e19: return "-INF"
    if abs(valor) < 1e-7: return "." 
    cadena = f"{valor:.{decimales}f}"
    return cadena.rstrip('0').rstrip('.') if decimales == 5 else cadena

def obtener_valor_seguro(objeto):
    return objeto.constant if hasattr(objeto, 'constant') else float(objeto)

# --- FUNCIÓN PRINCIPAL DE REPORTE AUTOMÁTICO ---
def generar_reporte_gams(mdl):
    """
    Toma cualquier modelo de Docplex, lo resuelve y genera el reporte estilo GAMS automáticamente.
    """
    solucion = mdl.solve()
    
    if not solucion:
        print("El modelo no encontró una solución factible o es ilimitado.")
        return
        
    cplex_inst = mdl.get_engine().get_cplex()
    
    # 1. Extraer elementos dinámicamente (no importa cuántos sean)
    variables = list(mdl.iter_variables())
    restricciones = list(mdl.iter_constraints())
    
    # Extraer sensibilidad usando la API interna
    sa_rhs = cplex_inst.solution.sensitivity.rhs()
    lim_inf_rhs, lim_sup_rhs = [r[0] for r in sa_rhs], [r[1] for r in sa_rhs]
    
    sa_obj = cplex_inst.solution.sensitivity.objective()
    lim_inf_obj, lim_sup_obj = [r[0] for r in sa_obj], [r[1] for r in sa_obj]
    
    # Extraer coeficientes actuales de la Función Objetivo
    expr_obj = mdl.get_objective_expr()
    coefs_actuales = [expr_obj.get_coef(var) if hasattr(expr_obj, 'get_coef') else 0.0 for var in variables]

    # 2. IMPRESIÓN DEL REPORTE
    print("\n--- Objective ranging... --- (Intervalos de recursos)")
    print(f"{'EQUATION NAME':<18} {'LOWER':>10} {'CURRENT':>10} {'UPPER':>10}")
    print("-" * 52)
    print(f"{'obj':<18} {'-INF':>10} {'NA':>10} {'+INF':>10}")
    
    for i, ct in enumerate(restricciones):
        nombre_ct = ct.name if ct.name else f"eq{i+1}"
        print(f"{nombre_ct:<18} {formatear_gams(lim_inf_rhs[i], 5):>10} "
              f"{formatear_gams(ct.rhs, 5):>10} {formatear_gams(lim_sup_rhs[i], 5):>10}")
        
    print("\n--- Variable ranging... --- (Intervalos para coeficientes F.O.)")
    print(f"{'VARIABLE NAME':<18} {'LOWER':>10} {'CURRENT':>10} {'UPPER':>10}")
    print("-" * 52)
    
    for i, var in enumerate(variables):
        nombre_var = var.name if var.name else f"var{i+1}"
        print(f"{nombre_var:<18} {formatear_gams(lim_inf_obj[i], 5):>10} "
              f"{formatear_gams(coefs_actuales[i], 5):>10} {formatear_gams(lim_sup_obj[i], 5):>10}")
    print(f"{'z':<18} {'-INF':>10} {'NA':>10} {'+INF':>10}")

    print(f"\nOptimal solution found\nObjective: {solucion.objective_value:.6f}\n")

    print(f"{'':<18} {'LOWER':>10} {'LEVEL':>10} {'UPPER':>10} {'MARGINAL':>10}")
    print("-" * 62)
    
    # Sección de Restricciones
    print(f"---- EQU {'obj':<9} {'-INF':>10} {'.':>10} {'.':>10} {'1.0000':>10}")
    for ct in restricciones:
        nombre_ct = ct.name[:9] if ct.name else f"eq{ct.index}"
        marginal   = formatear_gams(ct.dual_value)
        
        # Ajuste dinámico de los límites LOWER y UPPER según el tipo de restricción (<=, >=, ==)
        if ct.sense.name == 'GE': # >=
            lim_inf = formatear_gams(ct.rhs)
            lim_sup = "+INF"
            nivel_uso = formatear_gams(obtener_valor_seguro(ct.rhs) + ct.slack_value)
        elif ct.sense.name == 'EQ': # ==
            lim_inf = formatear_gams(ct.rhs)
            lim_sup = formatear_gams(ct.rhs)
            nivel_uso = formatear_gams(ct.rhs)
        else: # <= (Por defecto)
            lim_inf = "-INF"
            lim_sup = formatear_gams(ct.rhs)
            nivel_uso = formatear_gams(obtener_valor_seguro(ct.rhs) - ct.slack_value)
            
        print(f"---- EQU {nombre_ct:<9} {lim_inf:>10} {nivel_uso:>10} {lim_sup:>10} {marginal:>10}")

    print("\n")
    print(f"{'':<18} {'LOWER':>10} {'LEVEL':>10} {'UPPER':>10} {'MARGINAL':>10}")
    print("-" * 62)
    
    # Sección de Variables
    for var in variables:
        nombre_var = var.name[:9] if var.name else f"var{var.index}"
        limite_inf = formatear_gams(var.lb)
        limite_sup = formatear_gams(var.ub)
        nivel      = formatear_gams(var.solution_value)
        marginal   = formatear_gams(var.reduced_cost)
        print(f"---- VAR {nombre_var:<9} {limite_inf:>10} {nivel:>10} {limite_sup:>10} {marginal:>10}")
        
    print(f"---- VAR {'z':<9} {'-INF':>10} {formatear_gams(solucion.objective_value):>10} {'+INF':>10} {'.':>10}")

# Ejemplos

## Ejemplo 1

In [7]:
# 1. Creas tu modelo vacío
modelo = Model(name='Primal1')

# 2. Defines las variables
x1 = modelo.continuous_var(lb=0, name='x1')
x2 = modelo.continuous_var(lb=0, name='x2')

# 3. Agregas la Función Objetivo
modelo.maximize(30 * x1 + 20 * x2)

# 4. Agregas las Restricciones
modelo.add_constraint(2 * x1 + x2 <= 8, ctname='r1')
modelo.add_constraint(x1 + 3 * x2 <= 8, ctname='r2')

# 5. Llamas a la funcion de reporte
generar_reporte_gams(modelo)


--- Objective ranging... --- (Intervalos de recursos)
EQUATION NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
obj                      -INF         NA       +INF
r1                    2.66667          8         16
r2                          4          8         24

--- Variable ranging... --- (Intervalos para coeficientes F.O.)
VARIABLE NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
x1                    6.66667         30         40
x2                         15         20         90
z                        -INF         NA       +INF

Optimal solution found
Objective: 128.000000

                        LOWER      LEVEL      UPPER   MARGINAL
--------------------------------------------------------------
---- EQU obj             -INF          .          .     1.0000
---- EQU r1              -INF     8.0000     8.0000    14.0000
---- EQU r2              -INF     8.0000     8.0000     2.0

In [9]:
# 1. Creas tu modelo vacío
modelo2 = Model(name='Dual1')

# 2. Defines las variables
y1 = modelo2.continuous_var(lb=0, name='y1')
y2 = modelo2.continuous_var(lb=0, name='y2')

# 3. Agregas la Función Objetivo
modelo2.minimize(8 * y1 + 8 * y2)

# 4. Agregas las Restricciones
modelo2.add_constraint(2 * y1 + y2 >= 30, ctname='r1')
modelo2.add_constraint(y1 + 3 * y2 >= 20, ctname='r2')

# 5. Llamas a la funcion de reporte
generar_reporte_gams(modelo2)


--- Objective ranging... --- (Intervalos de recursos)
EQUATION NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
obj                      -INF         NA       +INF
r1                    6.66667         30         40
r2                         15         20         90

--- Variable ranging... --- (Intervalos para coeficientes F.O.)
VARIABLE NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
y1                    2.66667          8         16
y2                          4          8         24
z                        -INF         NA       +INF

Optimal solution found
Objective: 128.000000

                        LOWER      LEVEL      UPPER   MARGINAL
--------------------------------------------------------------
---- EQU obj             -INF          .          .     1.0000
---- EQU r1           30.0000    30.0000       +INF     3.2000
---- EQU r2           20.0000    20.0000       +INF     1.6

## Ejemplo 7

In [13]:
# 1. Creas tu modelo vacío
modelo = Model(name='Primal7')

# 2. Defines las variables
x1 = modelo.continuous_var(lb=0, name='x1')
x2 = modelo.continuous_var(lb=0, name='x2')

# 3. Agregas la Función Objetivo
modelo.maximize(3 * x1 + 5 * x2)

# 4. Agregas las Restricciones
modelo.add_constraint(x1 <= 4, ctname='r1')
modelo.add_constraint(2 * x2 <= 12, ctname='r2')
modelo.add_constraint(3 * x1 + 2 * x2 <= 18, ctname='r2')

# 5. Llamas a la funcion de reporte
generar_reporte_gams(modelo)


--- Objective ranging... --- (Intervalos de recursos)
EQUATION NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
obj                      -INF         NA       +INF
r1                          2          4       +INF
r2                          6         12         18
r2                         12         18         24

--- Variable ranging... --- (Intervalos para coeficientes F.O.)
VARIABLE NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
x1                          .          3        7.5
x2                          2          5       +INF
z                        -INF         NA       +INF

Optimal solution found
Objective: 36.000000

                        LOWER      LEVEL      UPPER   MARGINAL
--------------------------------------------------------------
---- EQU obj             -INF          .          .     1.0000
---- EQU r1              -INF     2.0000     4.0000          .
---- EQU

In [12]:
# 1. Creas tu modelo vacío
modelo = Model(name='Dual7')

# 2. Defines las variables
x1 = modelo.continuous_var(lb=0, name='x1')
x2 = modelo.continuous_var(lb=0, name='x2')

# 3. Agregas la Función Objetivo
modelo.maximize(3 * x1 + 5 * x2)

# 4. Agregas las Restricciones
modelo.add_constraint(x1 <= 4, ctname='r1')
modelo.add_constraint(2 * x2 <= 15, ctname='r2')
modelo.add_constraint(3 * x1 + 2 * x2 <= 15, ctname='r2')

# 5. Llamas a la funcion de reporte
generar_reporte_gams(modelo)


--- Objective ranging... --- (Intervalos de recursos)
EQUATION NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
obj                      -INF         NA       +INF
r1                          .          4       +INF
r2                         15         15       +INF
r2                          .         15         15

--- Variable ranging... --- (Intervalos para coeficientes F.O.)
VARIABLE NAME           LOWER    CURRENT      UPPER
----------------------------------------------------
x1                       -INF          3        7.5
x2                          2          5       +INF
z                        -INF         NA       +INF

Optimal solution found
Objective: 37.500000

                        LOWER      LEVEL      UPPER   MARGINAL
--------------------------------------------------------------
---- EQU obj             -INF          .          .     1.0000
---- EQU r1              -INF          .     4.0000          .
---- EQU